# SafeRoute-AI V3 — Pre-crash KSI Model

Trains an improved **pre-crash** binary KSI model without overwriting V1/V2.

| Split | Years |
|-------|-------|
| Train | 2009–2013 |
| Validation | 2014–2015 |
| Holdout (eval only) | 2016–2017 |

**Why V2 < V1:** V1 used post-crash vehicle-composition features. V2 removed them (correct for deployment) and lost signal.

**V3 approach:** keep only legitimate pre-crash inputs — map/road infrastructure + ego-vehicle/driver profile + engineered risk flags — then tune LightGBM.


In [ ]:
# Colab setup (skip locally if packages already installed)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install lightgbm joblib
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# Local defaults used for the Windows project run.
# On Colab, point these at your Drive dataset backups.
ACCIDENT_PATH = Path('..') / 'Accident_Information.csv.zip'
VEHICLE_PATH = Path('..') / 'Vehicle_Information.csv.zip'
OUT_DIR = Path('models')
METRICS_DIR = Path('outputs')

if IN_COLAB:
    ACCIDENT_PATH = Path('/content/drive/MyDrive/SafeRoute_AI_Datasets/Accident_Information.csv.zip')
    VEHICLE_PATH = Path('/content/drive/MyDrive/SafeRoute_AI_Datasets/Vehicle_Information.csv.zip')
    OUT_DIR = Path('/content/drive/MyDrive/SafeRoute_AI_Models')
    METRICS_DIR = Path('/content/drive/MyDrive/SafeRoute_AI_Models')

print('Accident path exists:', ACCIDENT_PATH.exists(), ACCIDENT_PATH)
print('Vehicle path exists:', VEHICLE_PATH.exists(), VEHICLE_PATH)


In [ ]:
# Runs the full V3 pipeline from the project script when available;
# otherwise paste/import train_saferoute_v3.py into the runtime.
import runpy
from pathlib import Path

script = Path('train_saferoute_v3.py')
if not script.exists():
    script = Path('/content/train_saferoute_v3.py')

assert script.exists(), (
    'Upload train_saferoute_v3.py to the runtime or set the local path.'
)

# Equivalent CLI:
# python train_saferoute_v3.py --accident-path ... --vehicle-path ...
import sys
sys.argv = [
    str(script),
    '--accident-path', str(ACCIDENT_PATH),
    '--vehicle-path', str(VEHICLE_PATH),
    '--out-dir', str(OUT_DIR),
    '--metrics-dir', str(METRICS_DIR),
]
runpy.run_path(str(script), run_name='__main__')


## Expected outputs

- saferoute_v3_precrash_2026.pkl — V3 checkpoint (does **not** overwrite V2)
- saferoute_v3_metrics.json — full validation/holdout metrics
- saferoute_v3_feature_importance.csv

Dashboard / simulator: leave on V2 until you explicitly switch the checkpoint path.
